# AIRA Model Fine-Tuning — QLoRA SFT on Google Colab
### Autonomous Infrastructure Resilience Architecture

This notebook implements the **Phase 4** SFT (Supervised Fine-Tuning) pipeline for **AIRA**. 
It uses standard **HuggingFace TRL + bitsandbytes QLoRA** training on a free-tier Google Colab T4 GPU to fine-tune `google/gemma-4-e4b-it` on our self-generated adversarial trajectory dataset (`sft_dataset.jsonl`).

At the end of training, it exports the fine-tuned LoRA adapter weights.

### 1. Install Unsloth and Dependencies

In [ ]:
%%capture
# Install standard HuggingFace SFT and bitsandbytes dependencies with transformers from source
!pip install git+https://github.com/huggingface/transformers.git
!pip install trl peft accelerate bitsandbytes datasets
!pip install pydantic structlog

### 2. Load Model and Tokenizer (4-bit Quantization)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "google/gemma-4-e4b-it"
max_seq_length = 4096 # Supports long Trivy cluster scans and multi-round battle context

print(f"[*] Loading model and tokenizer in Bfloat16 for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16, # Use bfloat16 to prevent GradScaler conflicts on multi-GPU
    device_map="auto"           # split layers across both GPUs safely
)
print("[SUCCESS] Model loaded in Bfloat16 precision!")

### 3. Configure LoRA Adapters (Rank 64 / Alpha 128)

In [ ]:
from peft import LoraConfig, get_peft_model

# Configure standard PEFT LoRA, targeting the inner .linear child modules inside Gemma 4's custom wrappers
peft_config = LoraConfig(
    r=64,  # LoRA Rank
    lora_alpha=128,  # LoRA Alpha
    target_modules=["q_proj.linear", "k_proj.linear", "v_proj.linear", "o_proj.linear", 
                    "gate_proj.linear", "up_proj.linear", "down_proj.linear"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

### 4. Load & Format Trajectory Dataset (`sft_dataset.jsonl`)

In [ ]:
# Upload your 'sft_dataset.jsonl' to the Colab files sidebar before running this cell
from datasets import load_dataset

dataset_path = "/kaggle/input/sft-dataset/sft_dataset.jsonl"
print(f"[*] Loading trajectory dataset from {dataset_path}...")
dataset = load_dataset("json", data_files=dataset_path, split="train")

# Format prompts dynamically using tokenizer's built-in chat template for Gemma 4
def format_prompts(examples):
    texts = []
    for messages in examples["messages"]:
        # SFT dataset has 'messages' list
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(text)
    return { "text" : texts }

# Remove all original columns (like 'messages') to leave ONLY the 'text' column
dataset = dataset.map(format_prompts, batched = True, remove_columns=dataset.column_names)
print(f"[SUCCESS] Dataset loaded! Formatted {len(dataset)} SFT samples. Columns: {dataset.column_names}")

### 5. Training Configuration (TRL SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = False,
        bf16 = True,
        gradient_checkpointing = True,
        logging_steps = 1,
        optim = "adamw_torch",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none"
    ),
)

In [ ]:
trainer_stats = trainer.train()

### 7. Export weights to 4-bit GGUF (for local Ollama deployment)

In [ ]:
# Save the fine-tuned LoRA adapter weights
model.save_pretrained("gemma-4-e4b-aira-lora")
tokenizer.save_pretrained("gemma-4-e4b-aira-lora")

# To deploy locally via Ollama:
# 1. Download your LoRA adapter folder from Colab.
# 2. Use a tool like llama.cpp to merge and quantize the model, or push the adapter directly to Hugging Face.